# Titanic Survival — EDA and ML Lifecycle Mapping

**Project 1:** Exploratory Data Analysis and mapping of the complete machine-learning lifecycle.

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

BASE_DIR = Path.cwd()
if not (BASE_DIR / "dataset").exists():
    BASE_DIR = BASE_DIR.parent

DATA_PATH = BASE_DIR / "dataset" / "titanic_train.csv"
OUTPUT_DIR = BASE_DIR / "outputs"
FIGURE_DIR = OUTPUT_DIR / "figures"
OUTPUT_DIR.mkdir(exist_ok=True)
FIGURE_DIR.mkdir(exist_ok=True)

df = pd.read_csv(DATA_PATH)
print("Dataset:", DATA_PATH)
print("Shape:", df.shape)

## 1. Initial Inspection

In [ ]:
display(df.head())
print("Shape:", df.shape)
print("\nColumns:", df.columns.tolist())
print("\nData types:")
display(df.dtypes)

## 2. Descriptive Statistics

In [ ]:
display(df.describe(include='all').T)

## 3. Missing-Value Analysis

In [ ]:
missing_summary = pd.DataFrame({
    "column": df.columns,
    "missing_count": df.isna().sum().values,
    "missing_percentage": (df.isna().mean().values * 100).round(2),
    "data_type": df.dtypes.astype(str).values
})
display(missing_summary)
missing_summary.to_csv(OUTPUT_DIR / "missing_summary.csv", index=False)

plt.figure(figsize=(8, 5))
plt.imshow(df.isna().astype(int).to_numpy(), aspect="auto", interpolation="nearest")
plt.title("Titanic Missing-Value Heatmap")
plt.xlabel("Columns")
plt.ylabel("Rows")
plt.xticks(range(len(df.columns)), df.columns, rotation=90)
plt.colorbar(label="Missing (1 = Yes)")
plt.show()

## 4. Duplicate-Row Check

In [ ]:
duplicate_summary = pd.DataFrame({
    "metric": ["total_rows", "duplicate_rows", "unique_rows"],
    "value": [len(df), int(df.duplicated().sum()), int(df.drop_duplicates().shape[0])]
})
display(duplicate_summary)
duplicate_summary.to_csv(OUTPUT_DIR / "duplicate_summary.csv", index=False)

## 5. Target Distribution

In [ ]:
target_distribution = (
    df["Survived"].value_counts().sort_index()
      .rename_axis("Survived").reset_index(name="count")
)
target_distribution["percentage"] = (
    target_distribution["count"] / len(df) * 100
).round(2)
display(target_distribution)
target_distribution.to_csv(OUTPUT_DIR / "target_distribution.csv", index=False)

plt.figure(figsize=(7, 5))
plt.bar(target_distribution["Survived"].astype(str), target_distribution["count"])
plt.title("Target Distribution: Survival")
plt.xlabel("Survived (0 = No, 1 = Yes)")
plt.ylabel("Passenger Count")
plt.show()

## 6. Survival by Sex

In [ ]:
survival_by_sex = (
    df.groupby("Sex", dropna=False)["Survived"]
      .agg(passengers="count", survivors="sum", survival_rate="mean")
      .reset_index()
)
survival_by_sex["survival_rate_percent"] = (survival_by_sex["survival_rate"] * 100).round(2)
display(survival_by_sex)
survival_by_sex.to_csv(OUTPUT_DIR / "survival_by_sex.csv", index=False)

plt.figure(figsize=(7, 5))
plt.bar(survival_by_sex["Sex"], survival_by_sex["survival_rate_percent"])
plt.title("Survival Rate by Sex")
plt.xlabel("Sex")
plt.ylabel("Survival Rate (%)")
plt.show()

## 7. Survival by Passenger Class

In [ ]:
survival_by_pclass = (
    df.groupby("Pclass", dropna=False)["Survived"]
      .agg(passengers="count", survivors="sum", survival_rate="mean")
      .reset_index()
)
survival_by_pclass["survival_rate_percent"] = (survival_by_pclass["survival_rate"] * 100).round(2)
display(survival_by_pclass)
survival_by_pclass.to_csv(OUTPUT_DIR / "survival_by_pclass.csv", index=False)

plt.figure(figsize=(7, 5))
plt.bar(survival_by_pclass["Pclass"].astype(str), survival_by_pclass["survival_rate_percent"])
plt.title("Survival Rate by Passenger Class")
plt.xlabel("Passenger Class")
plt.ylabel("Survival Rate (%)")
plt.show()

## 8. Survival by Age Group

In [ ]:
age_bins = pd.cut(
    df["Age"], bins=[0, 12, 18, 30, 50, 80],
    labels=["Child (0-12)", "Teen (13-18)", "Young Adult (19-30)",
            "Adult (31-50)", "Older Adult (51-80)"],
    include_lowest=True
)
survival_by_age_group = (
    pd.DataFrame({"Age_Group": age_bins, "Survived": df["Survived"]})
      .dropna()
      .groupby("Age_Group", observed=False)["Survived"]
      .agg(passengers="count", survivors="sum", survival_rate="mean")
      .reset_index()
)
survival_by_age_group["survival_rate_percent"] = (
    survival_by_age_group["survival_rate"] * 100
).round(2)
display(survival_by_age_group)
survival_by_age_group.to_csv(OUTPUT_DIR / "survival_by_age_group.csv", index=False)

plt.figure(figsize=(9, 5))
plt.bar(survival_by_age_group["Age_Group"].astype(str), survival_by_age_group["survival_rate_percent"])
plt.title("Survival Rate by Age Group")
plt.xlabel("Age Group")
plt.ylabel("Survival Rate (%)")
plt.xticks(rotation=20)
plt.show()

## 9. Correlation Analysis

In [ ]:
numeric_cols = df.select_dtypes(include=np.number).columns
corr = df[numeric_cols].corr().round(4)
display(corr)
corr.to_csv(OUTPUT_DIR / "correlation_matrix.csv")

plt.figure(figsize=(8, 6))
plt.imshow(corr.to_numpy(), aspect="auto", interpolation="nearest")
plt.title("Numeric Feature Correlation Matrix")
plt.xticks(range(len(corr.columns)), corr.columns, rotation=90)
plt.yticks(range(len(corr.index)), corr.index)
plt.colorbar(label="Correlation")
for i in range(corr.shape[0]):
    for j in range(corr.shape[1]):
        plt.text(j, i, f"{corr.iloc[i, j]:.2f}", ha="center", va="center")
plt.show()

## 10. Dataset Summary

In [ ]:
dataset_summary = pd.DataFrame({
    "metric": ["rows", "columns", "survivors", "non_survivors",
               "survival_rate_percent", "missing_cells", "duplicate_rows"],
    "value": [len(df), df.shape[1], int(df["Survived"].sum()),
              int((df["Survived"] == 0).sum()),
              round(df["Survived"].mean() * 100, 2),
              int(df.isna().sum().sum()), int(df.duplicated().sum())]
})
display(dataset_summary)
dataset_summary.to_csv(OUTPUT_DIR / "dataset_summary.csv", index=False)

## 11. ML Lifecycle Mapping

In [ ]:
lifecycle = pd.DataFrame([
    ["1", "Problem Definition", "Study passenger survival patterns and define the target.", "Survived"],
    ["2", "Data Acquisition", "Obtain the Titanic passenger training dataset.", "891 labeled records"],
    ["3", "Data Understanding", "Inspect shape, columns, types, statistics, and target.", "Data profile"],
    ["4", "Data Quality", "Check missing values and duplicate rows.", "Missing/duplicate summary"],
    ["5", "EDA", "Analyze survival by sex, class, age, and numeric relationships.", "EDA tables and plots"],
    ["6", "Preprocessing", "For predictive modeling, impute/encode using training-only transformations.", "Leakage-safe pipeline"],
    ["7", "Feature Engineering", "Potential features include FamilySize, IsAlone, Title, and FarePerPerson.", "Derived features"],
    ["8", "Modeling", "Potential baseline: Logistic Regression; compare tree models if required.", "Classification model"],
    ["9", "Evaluation", "Accuracy, precision, recall, F1, ROC-AUC, confusion matrix.", "Evaluation results"],
    ["10", "Interpretation", "Explain important relationships and model behavior.", "Interpretation"],
    ["11", "Deployment", "Package preprocessing and model together.", "Inference pipeline"],
    ["12", "Monitoring", "Monitor input quality and distribution/prediction drift.", "Monitoring checks"],
], columns=["phase", "lifecycle_stage", "activity", "expected_output"])
display(lifecycle)
lifecycle.to_csv(OUTPUT_DIR / "ml_lifecycle_mapping.csv", index=False)

## 12. Conclusion

The EDA establishes the dataset structure, missingness, target distribution, and major survival patterns. The lifecycle mapping shows how the analysis connects to preprocessing, feature engineering, modeling, evaluation, deployment, and monitoring.